# Module 02: Tools & Custom Tools

**Tools are the agent's hands.**

A tool is two things at once: a Python function the agent can *call*, and a schema the LLM can *read*. The LLM never sees your source code — it only sees the schema. It reads the schema to decide when to use the tool and how to fill in the arguments.

**Bad docstrings = confused agents.**

The anatomy of a tool schema:

```
Tool Schema:
  name:        "csv_summary"           ← used in LLM prompt
  description: "Loads a CSV..."        ← LLM decides when to use this
  inputs:      {"filepath": {...}}     ← LLM fills these arguments
  output_type: "string"               ← LLM knows what to expect back
```

Each field is a direct line of communication between you and the model. Treat them like documentation — be precise, give input format examples, and explain what the output looks like.

## Setup

In [ ]:
# Install required packages
# Uncomment the line below if running in Google Colab or a fresh environment
# !uv pip install smolagents python-dotenv duckduckgo-search mlflow
# Or using pip:
# !pip install smolagents python-dotenv duckduckgo-search mlflow

In [ ]:
import os

# ----- HF_TOKEN Setup -----
# Option A: Load from .env file (local development)
# from dotenv import load_dotenv
# load_dotenv()

# Option B: Google Colab Secrets
# Uncomment the lines below when running in Google Colab.
# Go to: Colab → Secrets (🔑 icon) → Add HF_TOKEN
# from google.colab import userdata
# os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# Option C: Set directly (not recommended for shared notebooks)
# os.environ['HF_TOKEN'] = 'hf_your_token_here'

In [ ]:
import os
from dotenv import load_dotenv
from smolagents import CodeAgent, InferenceClientModel, tool, Tool

load_dotenv()

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    token=os.environ["HF_TOKEN"],
)
print("Ready.")

## Built-in Tools

Before building your own, let's inspect what smolagents ships with.

In [ ]:
from smolagents import PythonInterpreterTool

python_tool = PythonInterpreterTool()

print("=== PythonInterpreterTool ===")
print(f"Name:        {python_tool.name}")
print(f"Description: {python_tool.description[:120]}...")
print(f"Inputs:      {list(python_tool.inputs.keys())}")
print(f"Output type: {python_tool.output_type}")

## The `@tool` Decorator

The simplest way to create a custom tool. Requirements:
- **Type hints** on all parameters and return value (mandatory — smolagents uses these to build the schema)
- **Docstring** with an `Args:` section describing each parameter
- **Return** a string, int, float, bool, or list

In [ ]:
from huggingface_hub import list_models

@tool
def top_hf_model(task: str) -> str:
    """
    Returns the most downloaded model for a given task on Hugging Face Hub.

    Args:
        task: The ML task name (e.g., 'text-classification', 'text-to-image',
              'text-generation', 'depth-estimation').
    """
    most_downloaded = next(iter(list_models(filter=task, sort="downloads", direction=-1)))
    return most_downloaded.id

# Inspect the generated schema
print("Name:", top_hf_model.name)
print("Description:", top_hf_model.description)
print("Inputs:", top_hf_model.inputs)
print("Output type:", top_hf_model.output_type)

In [ ]:
agent = CodeAgent(tools=[top_hf_model], model=model)
result = agent.run("What is the most downloaded text-generation model on HuggingFace?")
print("\nResult:", result)

## Subclassing `Tool`

Use this pattern when your tool needs:
- An `__init__` (e.g., to store an API key or load a file)
- Multiple helper methods
- Complex input validation

Implement the `forward()` method — that's where your tool logic lives.

In [ ]:
import pandas as pd

class CSVSummaryTool(Tool):
    name = "csv_summary"
    description = (
        "Loads a CSV file from a local path and returns summary statistics "
        "including shape, column names, dtypes, and describe() output. "
        "Use this to quickly understand the structure of a dataset."
    )
    inputs = {
        "filepath": {
            "type": "string",
            "description": "Absolute or relative path to the CSV file to analyze.",
        }
    }
    output_type = "string"

    def forward(self, filepath: str) -> str:
        df = pd.read_csv(filepath)
        summary = (
            f"Shape: {df.shape}\n"
            f"Columns: {list(df.columns)}\n"
            f"Dtypes:\n{df.dtypes.to_string()}\n"
            f"Stats:\n{df.describe().to_string()}"
        )
        return summary

# Test it directly before using with an agent (good practice!)
csv_tool = CSVSummaryTool()
print("Tool name:", csv_tool.name)
print("Tool inputs:", csv_tool.inputs)

## Tool Schema Deep-Dive

This is what smolagents sends to the LLM when it has your tool available. Notice: the quality of your description and input docs directly determines whether the agent uses your tool correctly.

In [ ]:
import json

# The tool's schema as seen by the LLM
schema = {
    "name": csv_tool.name,
    "description": csv_tool.description,
    "inputs": csv_tool.inputs,
    "output_type": csv_tool.output_type,
}
print(json.dumps(schema, indent=2))

## Exercises

In [ ]:
# TODO Exercise 1: Stats tool with @tool decorator
# Write a @tool function called `describe_numbers` that:
#   - Takes a single argument `numbers: str` (comma-separated values, e.g. "1,2,3,4,5")
#   - Returns a string with mean, median, and standard deviation
#   - Does NOT use the statistics library — compute manually or use sum()/len()
# 
# Then create a CodeAgent with this tool and ask:
# "What are the mean, median, and std dev of: 12, 45, 7, 89, 34, 56, 23?"

# Your code here:

In [ ]:
# TODO Exercise 2: CryptoPriceTool subclass
# Create a Tool subclass called CryptoPriceTool that:
#   - name = "crypto_price"
#   - Takes: coin_id: str (e.g., "bitcoin", "ethereum", "solana")
#   - Calls the free CoinGecko API:
#     https://api.coingecko.com/api/v3/simple/price?ids={coin_id}&vs_currencies=usd
#   - Returns a string like: "bitcoin: $67,234 USD"
#   - Handles errors gracefully (coin not found, API down)
#
# Hint: import requests

# Your code here:

In [ ]:
# TODO Exercise 3: Combined agent
# Combine BOTH tools from exercises 1 and 2 into a single CodeAgent.
# Ask it: "What is the mean and std dev of [3, 7, 1, 9, 4, 6]?
#          Also, what is the current price of ethereum?"
# 
# Observe: does the agent call both tools? In what order?
# Print the number of steps taken.

# Your code here:

## What You Built

You now know how to:
- Inspect built-in smolagents tools and understand their schema
- Create tools using the `@tool` decorator (fast, simple)
- Create tools by subclassing `Tool` (stateful, complex)
- Read and write tool schemas that the LLM will actually use correctly
- Integrate custom tools into a `CodeAgent`

**Key insight:** The tool's description is part of the LLM prompt. Treat it like documentation — be precise, give examples of inputs, explain what the output looks like.

## Next Module Preview

**Module 03: CodeAgent vs ToolCallingAgent**

You've been using `CodeAgent` exclusively. In Module 03 you'll meet `ToolCallingAgent` — a fundamentally different way agents reason. You'll run the same task with both and learn exactly when to use each. Plus: a 2-line demo of swapping HuggingFace for OpenAI.